In [1]:
import pandas as pd
import numpy as np

PRO02_CSV_PATH = '../pro02.csv'

df = pd.read_csv(PRO02_CSV_PATH, encoding='cp949') #utf-8 or cp949

df

,연도,월,연월,수출입구분,수출입구분설명,물동량
0,2003,2,2003-02,OT,수출환적,161074.50
1,2003,11,2003-11,IT,수입환적,170733.50
2,2006,1,2006-01,OT,수출환적,203348.25
3,2018,1,2018-01,OO,수출,413797.75
4,2013,6,2013-06,OO,수출,389650.25
...,...,...,...,...,...,...
1371,1996,1,1996-01,OO,수출,158518.00
1372,2001,6,2001-06,OO,수출,215325.25
1373,2001,9,2001-09,OT,수출환적,127883.00
1374,2004,1,2004-01,IT,수입환적,181137.50


In [2]:
pd.DataFrame({'자료형': df.dtypes.astype('str'),
              '비결측수': df.notna().sum(),
              '결측수': df.isna().sum(),
              '결측률(%)': df.isna().mean()*100,
              '고유값 수': df.nunique(dropna=True)})

,자료형,비결측수,결측수,결측률(%),고유값 수
연도,int64,1376,0,0.0,34
월,int64,1376,0,0.0,12
연월,str,1376,0,0.0,398
수출입구분,str,1376,0,0.0,4
수출입구분설명,str,1376,0,0.0,4
물동량,float64,1376,0,0.0,1374


In [3]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
연도,1376.0,2009.982558,8.861585,1992.0,2003.00,2010.0,2017.2500,2025.0
월,1376.0,6.470930,3.464399,1.0,3.00,6.0,9.0000,12.0
물동량,1376.0,324784.228924,119286.831230,90300.0,222387.25,322195.5,420758.4375,635437.0


In [7]:
data_cols = ['연도', '수출입구분설명','물동량'] #원하는 수치만 선택

project = df[data_cols]

project

,연도,수출입구분설명,물동량
0,2003,수출환적,161074.50
1,2003,수입환적,170733.50
2,2006,수출환적,203348.25
3,2018,수출,413797.75
4,2013,수출,389650.25
...,...,...,...
1371,1996,수출,158518.00
1372,2001,수출,215325.25
1373,2001,수출환적,127883.00
1374,2004,수입환적,181137.50


In [8]:
project['연도'].unique().tolist()

[2003,
 2006,
 2018,
 2013,
 2012,
 2017,
 2014,
 2015,
 2019,
 2022,
 1993,
 1995,
 2001,
 2004,
 2007,
 2010,
 2024,
 1994,
 1996,
 1997,
 1998,
 1999,
 2002,
 2005,
 2008,
 2025,
 2021,
 2009,
 2011,
 2016,
 1992,
 2020,
 2023,
 2000]

In [ ]:
# 수출환적 + 수입환적 → 환적으로 통합
temp = project.copy()
temp['구분'] = temp['수출입구분설명'].replace({
    '수출환적': '환적',
    '수입환적': '환적'
})

# 연도별 합계 계산
result_data = (
    temp.groupby(['연도', '구분'])['물동량']
        .sum()
        .unstack(fill_value=0)
        .reindex(columns=['수출', '수입', '환적'], fill_value=0)
        .reset_index()
)

result_data

구분,연도,수출,수입,환적
0,1992,1452695.00,1182679.00,0.00
1,1993,1540479.00,1400756.00,0.00
2,1994,1721176.00,1861341.00,0.00
3,1995,1912575.75,2177758.00,0.00
4,1996,2023501.00,2338534.00,0.00
5,1997,2196912.50,2614249.00,0.00
6,1998,2449246.75,2862064.75,0.00
7,1999,2471408.75,3189473.25,0.00
8,2000,2596600.75,3785717.00,0.00
9,2001,2558561.50,2571132.50,2943107.75


In [ ]:
# 연도 제외 수출 + 수입 + 환적 합계
result_data['합계'] = result_data[['수출', '수입', '환적']].sum(axis=1)

# 왼쪽 위의 '구분' 제거
result_data.columns.name = None

# 각 항목 비중
result_data['수출비중(%)'] = (result_data['수출'] / result_data['합계'] * 100).round(2)
result_data['수입비중(%)'] = (result_data['수입'] / result_data['합계'] * 100).round(2)
result_data['환적비중(%)'] = (result_data['환적'] / result_data['합계'] * 100).round(2)
result_data = result_data.set_index('연도')


result_data


,수출,수입,환적,합계,수출비중(%),수입비중(%),환적비중(%)
연도,,,,,,,
1992,1452695.00,1182679.00,0.00,2635374.00,55.12,44.88,0.00
1993,1540479.00,1400756.00,0.00,2941235.00,52.38,47.62,0.00
1994,1721176.00,1861341.00,0.00,3582517.00,48.04,51.96,0.00
1995,1912575.75,2177758.00,0.00,4090333.75,46.76,53.24,0.00
1996,2023501.00,2338534.00,0.00,4362035.00,46.39,53.61,0.00
1997,2196912.50,2614249.00,0.00,4811161.50,45.66,54.34,0.00
1998,2449246.75,2862064.75,0.00,5311311.50,46.11,53.89,0.00
1999,2471408.75,3189473.25,0.00,5660882.00,43.66,56.34,0.00
2000,2596600.75,3785717.00,0.00,6382317.75,40.68,59.32,0.00
